In [1]:
import json                                          # Standard library for JSON I/O :contentReference[oaicite:17]{index=17}
from pymatgen.core.structure import Structure         # PyMatGen’s Structure class :contentReference[oaicite:18]{index=18}
from monty.serialization import loadfn   
import os            # loadfn to read JSON easily (from monty) :contentReference[oaicite:19]{index=19}
import shutil
import numpy as np
import random
from pymatgen.core.structure import Structure

import pickle as pk

In [2]:
from monty.serialization import loadfn
import glob
import os

# ======================================================
# Base path to your dataset
# ======================================================
BASE_PATH = "/home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/data"

# ======================================================
# Function to analyze a folder of JSON files
# ======================================================
def analyze_json_folder(folder_path, category_name):
    json_files = sorted(glob.glob(os.path.join(folder_path, "*.json")))
    
    print(f"\n{'='*70}")
    print(f"Category: {category_name}")
    print(f"Path    : {folder_path}")
    print(f"{'-'*70}")

    category_count = 0

    for jf in json_files:
        data = loadfn(jf)
        n_structures = len(data)
        category_count += n_structures

        print(f"{os.path.basename(jf):40s} → {n_structures:5d} structures")

    print(f"{'-'*70}")
    print(f"TOTAL ({category_name}) → {category_count} structures")

    return category_count


# ======================================================
# Analyze all categories
# ======================================================
total_all = 0

total_all += analyze_json_folder(
    os.path.join(BASE_PATH, "pressure_disp"),
    "Pressure + Displacement"
)

total_all += analyze_json_folder(
    os.path.join(BASE_PATH, "pressure_strain_disp"),
    "Pressure + Strain + Displacement (CORE)"
)

total_all += analyze_json_folder(
    os.path.join(BASE_PATH, "strain_disp"),
    "Strain + Displacement"
)

total_all += analyze_json_folder(
    os.path.join(BASE_PATH, "disp_only"),
    "Displacement Only"
)

total_all += analyze_json_folder(
    os.path.join(BASE_PATH, "uniaxial_strain"),
    "Uniaxial Strain (X, Y, Z)"
)

# ======================================================
# Grand total
# ======================================================
print(f"\n{'='*70}")
print(f"GRAND TOTAL STRUCTURES IN DATASET → {total_all}")
print(f"{'='*70}")



Category: Pressure + Displacement
Path    : /home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/data/pressure_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp.json               →   250 structures
Mg3Bi2_3_GPa_0.3_disp.json               →   250 structures
Mg3Bi2_5_GPa_0.3_disp.json               →   250 structures
Mg3Bi2_7_GPa_0.3_disp.json               →   250 structures
----------------------------------------------------------------------
TOTAL (Pressure + Displacement) → 1000 structures

Category: Pressure + Strain + Displacement (CORE)
Path    : /home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/data/pressure_strain_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp_1_pct.json         →    50 structures
Mg3Bi2_1_GPa_0.3_disp_3_pct.json         →    50 structures
Mg3Bi2_1_GPa_0.3_disp_5_pct.json         →    50 structures
Mg3Bi2_1_GPa_0.3_disp_7_pct.json         →    50 structures


#### It will give the format for traininng used in MTP Potential, Here unit of stress is ev, that of energy , with train and val both in the ratio of 80/20 

In [8]:
# ============================================================
# JSON-wise balanced TRAIN / VAL CFG generator for MTP
# ============================================================

from monty.serialization import loadfn
from pymatgen.core import Structure
import glob
import os
import random

# ============================================================
# USER SETTINGS
# ============================================================
BASE_PATH = "/home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/data"
OUT_PATH  = "/home/ashwani/Mg3Bi2/"

VAL_FRACTION = 0.10        # 10% validation from EACH JSON
RANDOM_SEED  = 42

SPECIES_ORDER = ["Mg", "Bi", "Sb"]
GPA_TO_EV_A3 = 0.006241509

# ============================================================
# Fixed species → type mapping
# ============================================================
SPECIES_TO_TYPE = {sp: i for i, sp in enumerate(SPECIES_ORDER)}

# ============================================================
# Structure → CFG writer (aligned, MTP-safe)
# ============================================================
def structure_to_cfg_aligned(structure, energy, forces, stresses, species_to_type):
    lattice = structure.lattice.matrix
    positions = structure.cart_coords
    num_atoms = len(structure)

    cfg = "BEGIN_CFG\n"
    cfg += f" Size\n    {num_atoms}\n"
    cfg += " Supercell\n"
    for row in lattice:
        cfg += f"         {' '.join(f'{v:12.6f}' for v in row)}\n"

    cfg += (
        " AtomData:  id type       cartes_x      cartes_y      cartes_z"
        "           fx          fy          fz\n"
    )

    for i, (pos, force, site) in enumerate(
        zip(positions, forces, structure.sites), start=1
    ):
        sym = str(site.species.elements[0])
        atom_type = species_to_type[sym]

        cfg += (
            f"{i:13d}    {atom_type:<4d}"
            f"{pos[0]:12.6f} {pos[1]:12.6f} {pos[2]:12.6f}   "
            f"{force[0]:10.6f} {force[1]:10.6f} {force[2]:10.6f}\n"
        )

    cfg += " Energy\n"
    cfg += f"        {energy:.12f}\n"

    cfg += " PlusStress:  xx          yy          zz          yz          xz          xy\n"
    cfg += f"        {' '.join(f'{v:10.5f}' for v in stresses)}\n"

    cfg += " Feature   EFS_by\tDFT\n"
    cfg += "END_CFG\n"

    return cfg

# ============================================================
# Categories
# ============================================================
categories = [
    "pressure_disp",
    "pressure_strain_disp",
    "strain_disp",
    "disp_only",
    "uniaxial_strain",
]

random.seed(RANDOM_SEED)

train_cfg_blocks = []
val_cfg_blocks   = []

print("\n================ JSON-WISE TRAIN / VAL SPLIT ================\n")

# ============================================================
# JSON-wise split
# ============================================================
for category in categories:
    folder = os.path.join(BASE_PATH, category)
    json_files = sorted(glob.glob(os.path.join(folder, "*.json")))

    cat_total = cat_train = cat_val = 0

    print(f"Category: {category}")
    print("-" * 70)

    for jf in json_files:
        data = loadfn(jf)
        n_total = len(data)
        n_val = max(1, int(VAL_FRACTION * n_total))

        random.shuffle(data)

        val_part   = data[:n_val]
        train_part = data[n_val:]

        for d in train_part:
            volume = d["structure"].volume
            stress_ev = [s * GPA_TO_EV_A3 * volume for s in d["outputs"]["virial_stress"]]

            train_cfg_blocks.append(
                structure_to_cfg_aligned(
                    d["structure"],
                    d["outputs"]["energy"],
                    d["outputs"]["forces"],
                    stress_ev,
                    SPECIES_TO_TYPE
                )
            )

        for d in val_part:
            volume = d["structure"].volume
            stress_ev = [s * GPA_TO_EV_A3 * volume for s in d["outputs"]["virial_stress"]]

            val_cfg_blocks.append(
                structure_to_cfg_aligned(
                    d["structure"],
                    d["outputs"]["energy"],
                    d["outputs"]["forces"],
                    stress_ev,
                    SPECIES_TO_TYPE
                )
            )

        cat_total += n_total
        cat_train += len(train_part)
        cat_val   += len(val_part)

        print(
            f"{os.path.basename(jf):40s} | "
            f"total={n_total:4d} | val={len(val_part):4d} | train={len(train_part):4d}"
        )

    print("-" * 70)
    print(
        f"{category:25s} | "
        f"total={cat_total:4d} | val={cat_val:4d} | train={cat_train:4d}\n"
    )

# ============================================================
# Write CFG files
# ============================================================
train_file = os.path.join(OUT_PATH, "train.cfg")
val_file   = os.path.join(OUT_PATH, "val.cfg")

with open(train_file, "w") as f:
    f.writelines(train_cfg_blocks)

with open(val_file, "w") as f:
    f.writelines(val_cfg_blocks)

print("=====================================================")
print(f"FINAL TRAIN SET : {len(train_cfg_blocks)} structures")
print(f"FINAL VAL   SET : {len(val_cfg_blocks)} structures")
print("=====================================================")
print(f"✔ train.cfg → {train_file}")
print(f"✔ val.cfg   → {val_file}")
print("\n✅ DONE: JSON-wise balanced TRAIN/VAL CFGs for MTP\n")



================ JSON-WISE TRAIN / VAL SPLIT ================

Category: pressure_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
Mg3Bi2_3_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
Mg3Bi2_5_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
Mg3Bi2_7_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
----------------------------------------------------------------------
pressure_disp             | total=1000 | val= 100 | train= 900

Category: pressure_strain_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp_1_pct.json         | total=  50 | val=   5 | train=  45
Mg3Bi2_1_GPa_0.3_disp_3_pct.json         | total=  50 | val=   5 | train=  45
Mg3Bi2_1_GPa_0.3_disp_5_pct.json         | total=  50 | val=   5 | train=  45
Mg3Bi2_1_GPa_0.3_disp_7_pct.json         | total=  50 | va

#### It will give the XYZ format for traininng used to train the NEQUIP Potential "https://nequip.readthedocs.io/en/latest/guide/reference/conventions.html" Here the unit to stress is eV/A3

### It will give the XYZ format for traininng used in GPU MD with stress in ev/A3, it will give files test and train if you want or train only

In [7]:
# ============================================================
# JSON-wise balanced train/test XYZ generator
# (Each JSON contributes equally to test)
# ============================================================

from monty.serialization import loadfn
from pymatgen.core import Structure
import glob
import os
import random

# ============================================================
# USER SETTINGS
# ============================================================
BASE_PATH = "/home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/data"
TRAIN_XYZ = "/home/ashwani/Mg3Bi2/train.xyz"
TEST_XYZ  = "/home/ashwani/Mg3Bi2/test.xyz"

TEST_FRACTION = 0.1
RANDOM_SEED = 42

FLIP_STRESS_SIGN = True
GPA_TO_EV_PER_A3 = 1.0 / 160.21766208

# ============================================================
# Stress conversion
# ============================================================
def convert_voigt6_gpa_to_ev_per_a3(voigt6):
    factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
    σxx, σyy, σzz, σyz, σxz, σxy = voigt6
    return [
        σxx*factor, σxy*factor, σxz*factor,
        σxy*factor, σyy*factor, σyz*factor,
        σxz*factor, σyz*factor, σzz*factor,
    ]

# ============================================================
# XYZ writer
# ============================================================
def write_entry_xyz(f, struct: Structure, energy, forces, stress):
    n = len(struct)
    f.write(f"{n}\n")

    flat_lat = [c for row in struct.lattice.matrix for c in row]
    lat_str = " ".join(f"{v:.9f}" for v in flat_lat)

    header = (
        f'lattice="{lat_str}" '
        f'energy={energy:.12f} '
        f'stress="{" ".join(f"{x:.12f}" for x in stress)}" '
        'properties=species:S:1:pos:R:3:forces:R:3'
    )
    f.write(header + "\n")

    coords = struct.cart_coords
    for i, (fx, fy, fz) in enumerate(forces):
        sym = struct.sites[i].species.elements[0].symbol
        x, y, z = coords[i]
        f.write(
            f"{sym:<2s} "
            f"{x: .6f} {y: .6f} {z: .6f}   "
            f"{fx: .6f} {fy: .6f} {fz: .6f}\n"
        )

# ============================================================
# CATEGORY LIST
# ============================================================
categories = [
    "pressure_disp",
    "pressure_strain_disp",
    "strain_disp",
    "disp_only",
    "uniaxial_strain",
]

random.seed(RANDOM_SEED)

train_entries = []
test_entries  = []

print("\n================ CATEGORY-WISE (JSON-WISE) SPLIT ================\n")

category_summary = {}

# ============================================================
# JSON-wise splitting
# ============================================================
for category in categories:
    folder = os.path.join(BASE_PATH, category)
    json_files = sorted(glob.glob(os.path.join(folder, "*.json")))

    cat_total = cat_train = cat_test = 0

    print(f"Category: {category}")
    print("-" * 70)

    for jf in json_files:
        data = loadfn(jf)
        n_total = len(data)
        n_test = max(1, int(TEST_FRACTION * n_total))

        random.shuffle(data)

        test_part  = data[:n_test]
        train_part = data[n_test:]

        # collect entries
        for d in train_part:
            train_entries.append({
                "structure": d["structure"],
                "energy": d["outputs"]["energy"],
                "forces": d["outputs"]["forces"],
                "stress": convert_voigt6_gpa_to_ev_per_a3(
                    d["outputs"]["virial_stress"]
                )
            })

        for d in test_part:
            test_entries.append({
                "structure": d["structure"],
                "energy": d["outputs"]["energy"],
                "forces": d["outputs"]["forces"],
                "stress": convert_voigt6_gpa_to_ev_per_a3(
                    d["outputs"]["virial_stress"]
                )
            })

        cat_total += n_total
        cat_train += len(train_part)
        cat_test  += len(test_part)

        print(
            f"{os.path.basename(jf):40s} | "
            f"total={n_total:4d} | test={len(test_part):4d} | train={len(train_part):4d}"
        )

    category_summary[category] = (cat_total, cat_test, cat_train)

    print("-" * 70)
    print(
        f"{category:25s} | "
        f"total={cat_total:4d} | test={cat_test:4d} | train={cat_train:4d}\n"
    )

# ============================================================
# Final summary
# ============================================================
print("=====================================================")
print(f"FINAL TRAIN SET : {len(train_entries)} structures")
print(f"FINAL TEST  SET : {len(test_entries)} structures")
print("=====================================================\n")

# ============================================================
# Write XYZ files
# ============================================================
def write_xyz(filename, entries):
    with open(filename, "w") as f:
        for e in entries:
            write_entry_xyz(
                f,
                e["structure"],
                e["energy"],
                e["forces"],
                e["stress"]
            )
    print(f"✔ Wrote {len(entries)} structures → {filename}")

write_xyz(TRAIN_XYZ, train_entries)
write_xyz(TEST_XYZ,  test_entries)

print("\n✅ DONE: JSON-wise balanced train/test XYZ files generated\n")



================ CATEGORY-WISE (JSON-WISE) SPLIT ================

Category: pressure_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp.json               | total= 250 | test=  25 | train= 225
Mg3Bi2_3_GPa_0.3_disp.json               | total= 250 | test=  25 | train= 225
Mg3Bi2_5_GPa_0.3_disp.json               | total= 250 | test=  25 | train= 225
Mg3Bi2_7_GPa_0.3_disp.json               | total= 250 | test=  25 | train= 225
----------------------------------------------------------------------
pressure_disp             | total=1000 | test= 100 | train= 900

Category: pressure_strain_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp_1_pct.json         | total=  50 | test=   5 | train=  45
Mg3Bi2_1_GPa_0.3_disp_3_pct.json         | total=  50 | test=   5 | train=  45
Mg3Bi2_1_GPa_0.3_disp_5_pct.json         | total=  50 | test=   5 | train=  45
Mg3Bi2_1_GPa_0.3_disp_7_pct.json         | tot

###  It will give the format for traininng used in Deep-MD with train and test data generation, here the unit of virial is eV

In [6]:
# ============================================================
# DeepMD TRAIN / VAL dataset generator
# JSON-wise split, category folders preserved
# ============================================================

import os
import glob
import shutil
import random
import numpy as np
from monty.serialization import loadfn

# ============================================================
# PATHS (AS YOU REQUESTED)
# ============================================================
BASE_PATH = "/home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/data"
OUT_ROOT  = "/home/ashwani/Mg3Bi2/deepmd"

TRAIN_DIR = os.path.join(OUT_ROOT, "train")
VAL_DIR   = os.path.join(OUT_ROOT, "val")

# ============================================================
# SETTINGS
# ============================================================
VAL_FRACTION = 0.10
RANDOM_SEED = 42

EXPORT_RAW_FILES = True
FLIP_STRESS_SIGN = False
GPA_TO_EV_PER_A3 = 1.0 / 160.21766208

categories = [
    "disp_only",
    "pressure_disp",
    "pressure_strain_disp",
    "strain_disp",
    "uniaxial_strain",
]

# ============================================================
# RESET OUTPUT DIRS
# ============================================================
for d in [TRAIN_DIR, VAL_DIR]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

random.seed(RANDOM_SEED)

# ============================================================
# UTILITIES
# ============================================================
def convert_voigt6_gpa_to_ev(voigt6, volume):
    factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
    xx, yy, zz, yz, xz, xy = voigt6
    stress_evA3 = [
        xx*factor, xy*factor, xz*factor,
        xy*factor, yy*factor, yz*factor,
        xz*factor, yz*factor, zz*factor,
    ]
    return [s * volume for s in stress_evA3]  # → eV

def save_raw(array, fname):
    with open(fname, "w") as f:
        for row in array:
            f.write(" ".join(map(str, row)) + "\n")

# ============================================================
# WRITE ONE DEEPMD DATASET
# ============================================================
def write_deepmd_dataset(data_list, out_dir, dataset_name):
    if len(data_list) == 0:
        print(f"⚠️  Skipping empty dataset: {dataset_name}")
        return

    folder = os.path.join(out_dir, dataset_name)
    os.makedirs(folder, exist_ok=True)

    # DeepMD expects per-dataset type map
    symbols = sorted({site.specie.symbol
                      for d in data_list
                      for site in d["structure"].sites})
    mapping = {s: i for i, s in enumerate(symbols)}

    with open(os.path.join(folder, "type_map.raw"), "w") as f:
        for s in symbols:
            f.write(s + "\n")

    coords, box, energy, force, virial, types = [], [], [], [], [], []

    for d in data_list:
        struct = d["structure"]
        coords.append(struct.cart_coords.flatten())
        box.append(struct.lattice.matrix.flatten())
        energy.append([d["outputs"]["energy"]])
        force.append(np.array(d["outputs"]["forces"]).flatten())
        virial.append(
            convert_voigt6_gpa_to_ev(
                d["outputs"]["virial_stress"],
                struct.volume
            )
        )
        types.append([mapping[s.specie.symbol] for s in struct.sites])

    coords = np.array(coords, np.float32)
    box    = np.array(box, np.float32)
    energy = np.array(energy, np.float32)
    force  = np.array(force, np.float32)
    virial = np.array(virial, np.float32)

    # type.raw (from first structure – DeepMD convention)
    with open(os.path.join(folder, "type.raw"), "w") as f:
        for t in types[0]:
            f.write(str(t) + "\n")

    set_dir = os.path.join(folder, "set.000")
    os.makedirs(set_dir, exist_ok=True)

    np.save(os.path.join(set_dir, "coord.npy"), coords)
    np.save(os.path.join(set_dir, "box.npy"), box)
    np.save(os.path.join(set_dir, "energy.npy"), energy)
    np.save(os.path.join(set_dir, "force.npy"), force)
    np.save(os.path.join(set_dir, "virial.npy"), virial)

    if EXPORT_RAW_FILES:
        save_raw(coords,  os.path.join(set_dir, "coord.raw"))
        save_raw(box,     os.path.join(set_dir, "box.raw"))
        save_raw(energy,  os.path.join(set_dir, "energy.raw"))
        save_raw(force,   os.path.join(set_dir, "force.raw"))
        save_raw(virial,  os.path.join(set_dir, "virial.raw"))

# ============================================================
# MAIN LOOP — CATEGORY + JSON-WISE SPLIT
# ============================================================
print("\n================ JSON-WISE TRAIN / VAL (DEEPMD) ================\n")

for category in categories:
    folder = os.path.join(BASE_PATH, category)
    json_files = sorted(glob.glob(os.path.join(folder, "*.json")))

    cat_train, cat_val = [], []

    print(f"Category: {category}")
    print("-" * 70)

    for jf in json_files:
        data = loadfn(jf)
        n_total = len(data)

        # ensure at least one train if possible
        n_val = max(1, int(VAL_FRACTION * n_total))
        if n_total == 1:
            n_val = 0

        random.shuffle(data)

        val_part   = data[:n_val]
        train_part = data[n_val:]

        cat_train.extend(train_part)
        cat_val.extend(val_part)

        print(
            f"{os.path.basename(jf):40s} | "
            f"total={n_total:4d} | val={len(val_part):4d} | train={len(train_part):4d}"
        )

    write_deepmd_dataset(cat_train, TRAIN_DIR, category)
    write_deepmd_dataset(cat_val,   VAL_DIR,   category)

    print("-" * 70)
    print(
        f"{category:25s} | "
        f"train={len(cat_train):4d} | val={len(cat_val):4d}\n"
    )

print("=====================================================")
print("✅ DONE: DeepMD TRAIN / VAL datasets created correctly")
print("=====================================================\n")



================ JSON-WISE TRAIN / VAL (DEEPMD) ================

Category: disp_only
----------------------------------------------------------------------
Mg3Bi2_pct_0.2_disp.json                 | total= 500 | val=  50 | train= 450
Mg3Bi2_pct_0.3_disp.json                 | total= 500 | val=  50 | train= 450
----------------------------------------------------------------------
disp_only                 | train= 900 | val= 100

Category: pressure_disp
----------------------------------------------------------------------
Mg3Bi2_1_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
Mg3Bi2_3_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
Mg3Bi2_5_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
Mg3Bi2_7_GPa_0.3_disp.json               | total= 250 | val=  25 | train= 225
----------------------------------------------------------------------
pressure_disp             | train= 900 | val= 100

Category: pressure_strain_disp
----

### GAP Potential, here the unit of stress is eV/A3